In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 9.9 MB/s eta 0:00:00


In [2]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [6]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0 , np.nan)
df.fillna(df.mean() , inplace = True)
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [7]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test) # we dont fit just because of the fear of Data Leakage

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [20]:
from sklearn.ensemble import RandomForestClassifier # A bagging Technique
from sklearn.model_selection import cross_val_score

def objective(trial) :
  n_estimators = trial.suggest_int('n_estimators' , 50 , 200)
  max_depth = trial.suggest_int('max_depth' , 3 , 20)
  model = RandomForestClassifier(n_estimators = n_estimators , max_depth = max_depth , random_state=42) # here random state is a seed
  result = cross_val_score(model , X_train , y_train , cv= 3, scoring = 'accuracy').mean()
  return result


In [23]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=30)


[I 2025-09-14 02:27:38,559] A new study created in memory with name: no-name-d52eb23d-5b9f-4ecc-8854-3142b23c9770
[I 2025-09-14 02:27:39,007] Trial 0 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 76, 'max_depth': 9}. Best is trial 0 with value: 0.7653631284916201.
[I 2025-09-14 02:27:39,714] Trial 1 finished with value: 0.7821229050279329 and parameters: {'n_estimators': 117, 'max_depth': 15}. Best is trial 1 with value: 0.7821229050279329.
[I 2025-09-14 02:27:40,220] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 85, 'max_depth': 19}. Best is trial 1 with value: 0.7821229050279329.
[I 2025-09-14 02:27:40,715] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 92, 'max_depth': 4}. Best is trial 1 with value: 0.7821229050279329.
[I 2025-09-14 02:27:41,243] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 93, 'max_depth': 7}. Best is trial 1 with value: 0.78212290502

In [13]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 129, 'max_depth': 16}


In [22]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420856
Best hyperparameters: {'n_estimators': 124, 'max_depth': 7}


In [24]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 117, 'max_depth': 15}


In [14]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params , random_state = 42)
best_model.fit(X_train , y_train)
y_pred = best_model.predict(X_test)
score = accuracy_score(y_test , y_pred)
print("SCORE IS :::: " , score)

SCORE IS ::::  0.7445887445887446


# **Using Random Sampling in Optuna**
## But Random Sampling is a bit less Optimized Technique when comparing to Bayesian Technique That is used in Tree-Structured Parzen Estimator

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score


In [16]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-09-14 02:22:38,824] A new study created in memory with name: no-name-26d16fb5-06e7-4aa7-b4bf-8c8fefa7935a
[I 2025-09-14 02:22:40,528] Trial 0 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 196, 'max_depth': 18}. Best is trial 0 with value: 0.7746741154562384.
[I 2025-09-14 02:22:41,005] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 64, 'max_depth': 17}. Best is trial 0 with value: 0.7746741154562384.
[I 2025-09-14 02:22:41,649] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 64, 'max_depth': 20}. Best is trial 0 with value: 0.7746741154562384.
[I 2025-09-14 02:22:42,487] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 83, 'max_depth': 12}. Best is trial 0 with value: 0.7746741154562384.
[I 2025-09-14 02:22:44,082] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 185, 'max_depth': 17}. Best is trial 0 with value: 0.7746741

In [17]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 50, 'max_depth': 7}


# **Optuna Visualization Techniques For Step By Step Investigation System**

In [18]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [19]:
plot_optimization_history(study).show()

In [26]:
plot_parallel_coordinate(study).show()

In [27]:
plot_slice(study).show()

In [28]:
plot_contour(study).show()

In [29]:
plot_param_importances(study).show()

# **Using ML Algorithms as HyperParameters (Seems Ridiculous)**

In [30]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [31]:
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [32]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-09-14 02:36:24,228] A new study created in memory with name: no-name-f0e9482f-4a0a-49d9-8f46-cf18c25c294e
[I 2025-09-14 02:36:26,591] Trial 0 finished with value: 0.7560521415270017 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 106, 'learning_rate': 0.18374070456626299, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.7560521415270017.
[I 2025-09-14 02:36:26,631] Trial 1 finished with value: 0.7262569832402234 and parameters: {'classifier': 'SVM', 'C': 0.3178046663751258, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.7560521415270017.
[I 2025-09-14 02:36:27,826] Trial 2 finished with value: 0.7690875232774674 and parameters: {'classifier': 'RandomForest', 'n_estimators': 209, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 5, 'bootstrap': True}. Best is trial 2 with value: 0.7690875232774674.
[I 2025-09-14 02:36:28,746] Trial 3 finished with value: 0.7690875232774674 and paramete

In [33]:
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.1358316913247658, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [34]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.756052,2025-09-14 02:36:24.231022,2025-09-14 02:36:26.590999,0 days 00:00:02.359977,NaN,NaN,GradientBoosting,NaN,NaN,0.183741,8.0,7.0,4.0,106.0,COMPLETE
1,1,0.726257,2025-09-14 02:36:26.591925,2025-09-14 02:36:26.631146,0 days 00:00:00.039221,0.317805,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.769088,2025-09-14 02:36:26.632692,2025-09-14 02:36:27.826564,0 days 00:00:01.193872,NaN,True,RandomForest,NaN,NaN,NaN,16.0,5.0,7.0,209.0,COMPLETE
3,3,0.769088,2025-09-14 02:36:27.827546,2025-09-14 02:36:28.746164,0 days 00:00:00.918618,NaN,False,RandomForest,NaN,NaN,NaN,7.0,10.0,5.0,207.0,COMPLETE
4,4,0.726257,2025-09-14 02:36:28.747202,2025-09-14 02:36:28.780401,0 days 00:00:00.033199,2.257242,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.754190,2025-09-14 02:37:17.644540,2025-09-14 02:37:19.167746,0 days 00:00:01.523206,NaN,NaN,GradientBoosting,NaN,NaN,0.098493,5.0,7.0,3.0,184.0,COMPLETE
96,96,0.763501,2025-09-14 02:37:19.168750,2025-09-14 02:37:19.208059,0 days 00:00:00.039309,0.202834,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.789572,2025-09-14 02:37:19.208996,2025-09-14 02:37:19.239042,0 days 00:00:00.030046,0.149594,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.789572,2025-09-14 02:37:19.240009,2025-09-14 02:37:19.276324,0 days 00:00:00.036315,0.123476,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [36]:
! pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 3.3 MB/s eta 0:00:00


In [37]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2025-09-14 02:40:04,127] A new study created in memory with name: no-name-ac09b7f3-fcbf-4645-959f-dbdf424e3c70


[0]	train-mlogloss:0.91530	eval-mlogloss:0.90431
[1]	train-mlogloss:0.75021	eval-mlogloss:0.71858
[2]	train-mlogloss:0.59189	eval-mlogloss:0.55062
[3]	train-mlogloss:0.47614	eval-mlogloss:0.42563
[4]	train-mlogloss:0.40248	eval-mlogloss:0.34981
[5]	train-mlogloss:0.34036	eval-mlogloss:0.28371
[6]	train-mlogloss:0.30145	eval-mlogloss:0.24567
[7]	train-mlogloss:0.29124	eval-mlogloss:0.23478
[8]	train-mlogloss:0.28994	eval-mlogloss:0.23429
[9]	train-mlogloss:0.26164	eval-mlogloss:0.20161
[10]	train-mlogloss:0.25533	eval-mlogloss:0.19154
[11]	train-mlogloss:0.25529	eval-mlogloss:0.19224
[12]	train-mlogloss:0.25470	eval-mlogloss:0.19192
[13]	train-mlogloss:0.25576	eval-mlogloss:0.19600
[14]	train-mlogloss:0.25498	eval-mlogloss:0.19458
[15]	train-mlogloss:0.25397	eval-mlogloss:0.19335
[16]	train-mlogloss:0.25297	eval-mlogloss:0.18940
[17]	train-mlogloss:0.25220	eval-mlogloss:0.18906
[18]	train-mlogloss:0.24422	eval-mlogloss:0.17882
[19]	train-mlogloss:0.24429	eval-mlogloss:0.17849
[20]	train

[I 2025-09-14 02:40:05,095] Trial 0 finished with value: 1.0 and parameters: {'lambda': 1.2250904678513507e-07, 'alpha': 3.6114636955552215e-07, 'eta': 0.23828434454277586, 'gamma': 0.023318103669313976, 'max_depth': 7, 'min_child_weight': 6, 'subsample': 0.4588392809168451, 'colsample_bytree': 0.5177577376554756}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.93739	eval-mlogloss:0.93529
[1]	train-mlogloss:0.75798	eval-mlogloss:0.74707
[2]	train-mlogloss:0.62329	eval-mlogloss:0.60578
[3]	train-mlogloss:0.52013	eval-mlogloss:0.50116
[4]	train-mlogloss:0.44022	eval-mlogloss:0.41806
[5]	train-mlogloss:0.37466	eval-mlogloss:0.34748
[6]	train-mlogloss:0.32293	eval-mlogloss:0.29446
[7]	train-mlogloss:0.28324	eval-mlogloss:0.25077
[8]	train-mlogloss:0.25168	eval-mlogloss:0.21511
[9]	train-mlogloss:0.22790	eval-mlogloss:0.18721
[10]	train-mlogloss:0.20877	eval-mlogloss:0.16456
[11]	train-mlogloss:0.19643	eval-mlogloss:0.14894
[12]	train-mlogloss:0.18711	eval-mlogloss:0.13964
[13]	train-mlogloss:0.18142	eval-mlogloss:0.13153
[14]	train-mlogloss:0.17736	eval-mlogloss:0.12583
[15]	train-mlogloss:0.17421	eval-mlogloss:0.12081
[16]	train-mlogloss:0.17139	eval-mlogloss:0.11646
[17]	train-mlogloss:0.16745	eval-mlogloss:0.11219
[18]	train-mlogloss:0.16445	eval-mlogloss:0.10727
[19]	train-mlogloss:0.16432	eval-mlogloss:0.10710
[20]	train

[I 2025-09-14 02:40:05,674] Trial 1 finished with value: 1.0 and parameters: {'lambda': 2.9410724283205743e-08, 'alpha': 3.716479641474016e-05, 'eta': 0.18529849999045672, 'gamma': 0.00041824437326498274, 'max_depth': 9, 'min_child_weight': 7, 'subsample': 0.898329432056787, 'colsample_bytree': 0.7470722684104807}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.92616	eval-mlogloss:0.91464


[I 2025-09-14 02:40:26,338] Trial 48 pruned. Trial was pruned at iteration 1.


[0]	train-mlogloss:1.06906	eval-mlogloss:1.06721
[1]	train-mlogloss:1.04050	eval-mlogloss:1.03469
[2]	train-mlogloss:1.01340	eval-mlogloss:1.00800
[3]	train-mlogloss:0.98247	eval-mlogloss:0.97495
[4]	train-mlogloss:0.95393	eval-mlogloss:0.94552


[I 2025-09-14 02:40:26,491] Trial 49 pruned. Trial was pruned at iteration 4.


Best trial: {'lambda': 1.2250904678513507e-07, 'alpha': 3.6114636955552215e-07, 'eta': 0.23828434454277586, 'gamma': 0.023318103669313976, 'max_depth': 7, 'min_child_weight': 6, 'subsample': 0.4588392809168451, 'colsample_bytree': 0.5177577376554756}
Best accuracy: 1.0
